Your task is to create a bert-base-classifier of vacancy areas based on their titles.

Each vacancy can have more than one area so it's **Multi-label classification** not Multiclass classification




In [12]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from nltk.tokenize import word_tokenize
from string import punctuation
from tqdm import tqdm

In [13]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [14]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler, Dataset, SequentialSampler
import random
import transformers

# Try two or more different bert-like models(different berts, robertas etc. or any other transformer based model) (**2 points max**)
 your notebook should contain the training process of all your models!

In [15]:
MODEL_NAME = 'bert-base-uncased'          # you can swap: 'distilbert-base-uncased', 'albert-base-v2', etc.
MAX_SEQ_LENGTH = 128                      # 512 is BERT hard-limit; 128 balances speed & quality for short job titles
RESULT_MODEL_PATH = './model.pt'

In [16]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed = 12
seed_everything(seed)

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [18]:
device

device(type='cuda')

In [19]:
punctuation = set('!"$%&\'()*,-/:;<=>?@[\\]^_`{|}~') # убрал #

In [20]:
def clean(text):
    return ' '.join([token.lower() for token in word_tokenize(text) if token not in punctuation])

In [21]:
df = pd.read_csv('/content/dataset_2020.csv')
df.shape

(78909, 2)

Each vacancy can have more than one area separated be space

Exapmle:

Malware Analyst for Imunify Security,analyst it_security

In [22]:
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)

# Finish TextClassificationDataset (**1 point max**)

In [23]:
class TextClassificationDataset(Dataset):
    def __init__(self, data, tokenizer, binarizer):
        self.data = data
        self.tokenizer = tokenizer
        sentences = [clean(sent) for sent in data.title.tolist()]
        self.target = [labels.split() for labels in data.area.tolist()]
        self.binarizer = binarizer
        self.target_one_hot = torch.tensor(self.binarizer.transform(self.target), dtype=torch.float)
        # tokenize once at init (fast)
        self.encodings = self.tokenizer(
            sentences,
            truncation=True,
            padding='max_length',
            max_length=MAX_SEQ_LENGTH,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.target_one_hot)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.target_one_hot[idx]
        return item

In [24]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
binarizer = MultiLabelBinarizer()
labels_train = [labels.split() for labels in df_train.area.tolist()]
binarizer.fit(labels_train)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

MultiLabelBinarizer()

In [25]:
batch_size = 8 # ToDo

train_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
train_sampler = RandomSampler(train_dataset)
train_dataloader =  DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size,)

valid_dataset = TextClassificationDataset(df_valid, tokenizer, binarizer)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)

test_dataset = TextClassificationDataset(df_test, tokenizer, binarizer)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


In [26]:
class BertForMultilabel(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()
        self.bert = transformers.BertModel.from_pretrained(MODEL_NAME)

        # TODO: add your custom layers here
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def train_bert(self, train_bert_flag=True):
        for param in self.bert.parameters():
            param.requires_grad = train_bert_flag

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        # TODO: implement forward pass
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

In [27]:
num_labels = len(binarizer.classes_)
model = BertForMultilabel(num_labels)
model.to(device)
;

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

''

# Train your classifier with freezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch


In [28]:
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0.0

    for batch in iterator:
        # move tensors to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # zero grads
        optimizer.zero_grad()

        # forward
        logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # compute loss
        loss = criterion(logits, labels)

        # backward
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [29]:
def validate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0.0

    with torch.no_grad():
        for batch in iterator:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [30]:
def logits_to_labels(logits):
    preds = nn.Sigmoid()(logits.view(-1, num_labels))
    preds = preds.to('cpu').numpy()>0.5
    return preds.tolist()

In [31]:
model.train_bert(False)

In [32]:
epochs = 3 # ToDo
criterion = nn.BCEWithLogitsLoss()# ToDo what criterion do you need for multilabel classification?
optimizer  = torch.optim.Adam(model.parameters(), lr=2e-5) # ToDo use adam optimizer
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1) # ToDo use StepLR scheduler

In [33]:
# ToDo Train your model
# ---------- training with early-stopping / best-val-loss save ----------
best_val_loss = float('inf')

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")

    # training pass
    train_loss = train(model, train_dataloader, optimizer, criterion)
    print(f"  train loss: {train_loss:.4f}")

    # validation pass
    val_loss = validate(model, valid_dataloader, criterion)
    print(f"  val   loss: {val_loss:.4f}")

    # Step the scheduler
    scheduler.step()

    # save checkpoint if validation improved
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH)
        print("  🎯  New best val loss -> model saved")

print("Training finished. Lowest val loss:", best_val_loss)

Epoch 1/3
  train loss: 0.1382
  val   loss: 0.0796
  🎯  New best val loss -> model saved
Epoch 2/3
  train loss: 0.0779
  val   loss: 0.0648
  🎯  New best val loss -> model saved
Epoch 3/3
  train loss: 0.0711
  val   loss: 0.0637
  🎯  New best val loss -> model saved
Training finished. Lowest val loss: 0.06367640454492003


In [34]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device)))
model.eval()  # inference mode

# run the test set through the model to get logits
test_logits = []
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        test_logits.append(logits)

test_logits = torch.cat(test_logits, dim=0)  # shape: (N_samples, num_labels)

# convert logits → binary predictions
test_preds = logits_to_labels(test_logits)
print("Test predictions shape:", len(test_preds), "samples")

Test predictions shape: 7891 samples


In [35]:
print(classification_report(
    binarizer.transform(test_dataset.target),  # ground-truth multi-hot
    test_preds,                                # predicted multi-hot
    target_names=binarizer.classes_))

                 precision    recall  f1-score   support

          admin       0.00      0.00      0.00        61
        analyst       1.00      0.16      0.27       302
    architector       0.00      0.00      0.00       111
      assistant       0.00      0.00      0.00        14
     consultant       0.00      0.00      0.00        23
          coord       0.00      0.00      0.00        11
  data_engineer       0.00      0.00      0.00       136
 data_scientist       0.00      0.00      0.00       154
       designer       1.00      0.00      0.01       409
devel_metodolog       0.00      0.00      0.00        44
         devops       0.00      0.00      0.00       338
       director       0.00      0.00      0.00        17
     doc_writer       0.00      0.00      0.00        18
    it_security       0.00      0.00      0.00        54
machine_learner       0.00      0.00      0.00        42
        manager       0.00      0.00      0.00       427
       networks       0.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Train your classifier with unfreezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch

In [36]:
# unfreeze BERT → full fine-tuning
model.train_bert(train_bert_flag=True)

epochs = 4          # 3-5 is usually enough for full fine-tuning
lr = 2e-5           # standard BERT fine-tune LR
WARMUP_PROPORTION = 0.1
warmup_steps = int(len(train_dataloader) * epochs * WARMUP_PROPORTION)
print(f"Warm-up steps: {warmup_steps}")

Warm-up steps: 2840


In [37]:
model.train_bert(True)

In [38]:
# no decay on bias and LayerNorm weight (standard BERT trick)
no_decay = ['bias', 'LayerNorm.weight']

param_optimizer = list(model.named_parameters())
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.001},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]

criterion = nn.BCEWithLogitsLoss()       # multi-label loss
lr = 2e-5                                # learning rate

# FIX: Use torch.optim.AdamW instead of transformers.optimization.AdamW
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=lr)

# Calculate total training steps
t_total = len(train_dataloader) * epochs

scheduler = transformers.optimization.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=t_total
)

In [ ]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device)))
test_preds = validate(model, test_dataloader, criterion)

In [ ]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

In [ ]:
# Results

# Results (3 points max)

Write your conclusion

What models and what training parameters did you use?

What was the reason for your choice?

What were the results?

What metrics do you consider the most important?